In [1]:
import datasets
from utils import GrazieProvider, ManimProvider, extract_json
from grazie.api.client.gateway import AuthType, GrazieApiGatewayClient, GrazieAgent
from grazie.api.client.endpoints import GrazieApiGatewayUrls

In [2]:
with open("./token.secret", 'r') as t: token = t.read()

client = GrazieApiGatewayClient(
    grazie_agent=GrazieAgent(name="grazie-api-gateway-client-readme", version="dev"),
    url=GrazieApiGatewayUrls.STAGING,
    grazie_jwt_token=token,
    auth_type=AuthType.USER,
)

In [18]:
provider = GrazieProvider(client, model="openai-gpt-4o")
ds = datasets.load_from_disk("subset")

In [19]:
ds

Dataset({
    features: ['value', 'definition', 'metaphor'],
    num_rows: 14
})

In [53]:
term = ds[4]
metaphor = term["metaphor"]
term

{'value': 'replace',
 'definition': 'The replace function in Kotlin is used to replace occurrences of a substring with another substring.',
 'metaphor': 'Imagine you have a book with a story that mentions a character named "John" many times. One day, the author decides to change the character\'s name to "Jack." \n\nYou (the replace function) are like an editor with a special task. You go through the entire book, and every time you see "John" (the substring to be replaced), you cross it out and write "Jack" (the new substring) in its place. \n\nBy the time you\'re done, every mention of "John" has been replaced with "Jack," and the story now reads with the new name throughout.'}

In [54]:
classes = provider.get_classes(term, metaphor)

In [55]:
classes_dict = extract_json(classes)

In [56]:
classes_dict['elements']

[{'name': 'Editor',
  'role': 'Represents the replace function, tasked with replacing occurrences of a substring in the book.',
  'actions': ['Goes through the book',
   "Finds occurrences of the old substring ('John')",
   "Replaces 'John' with 'Jack'"],
  'code': 'from manim import *\n\nclass Editor(VGroup):\n    def __init__(self, editor_height=1.0, start_position=ORIGIN, colour=BLUE, **kwargs):\n        super().__init__(**kwargs)\n        self.editor_height = editor_height\n        # Create the head\n        self.head = Circle(radius=self.editor_height * 0.3)\n        self.head.set_color(colour)\n        self.head.set_fill(colour, opacity=1)\n        # Create the body\n        self.body = Rectangle(width=self.editor_height * 0.7, height=self.editor_height)\n        self.body.set_color(colour)\n        self.body.set_fill(colour, opacity=1)\n        # Position the head above the body\n        self.head.next_to(self.body, UP, buff=0.1)\n        # Move the editor to the starting positi

In [57]:
import os, subprocess

for i in range(len(classes_dict['elements'])):
    open("/tmp/dummy.py", "w").write(
        f"{classes_dict['elements'][i]['code']}"
    )
    command = [os.path.join("pylint"), "-E", "/tmp/dummy.py"]
    process = subprocess.run(command, capture_output=True, text=True)
    static_errors = process.stdout
    print(f"{i}: {static_errors}")

0: ************* Module dummy
/tmp/dummy.py:28:38: E0602: Undefined variable 'TransformMatchingText' (undefined-variable)

1: ************* Module dummy
/tmp/dummy.py:8:19: E0602: Undefined variable 'Page' (undefined-variable)

2: ************* Module dummy
/tmp/dummy.py:8:19: E0602: Undefined variable 'LineOfText' (undefined-variable)

3: 


In [58]:
desc = provider.get_description(term, metaphor, str(classes_dict))
# desc = ""
manim_code = provider.get_manim(term, metaphor, str(classes_dict), desc)

In [59]:
manim_provider = ManimProvider(provider, term, working_dir="/home/ynoviello/PycharmProjects/AI_Metaphors/manim_stuff")

In [61]:
manim_provider.write_python(manim_code)
error = manim_provider.execute_manim_script()

In [51]:
import json
with open("/home/ynoviello/PycharmProjects/AI_Metaphors/manim_stuff/scripts/best-scripts/replace.json", 'w') as j:
    json.dump(classes_dict, j)

In [62]:
print(error)


Animation 0: Write(Text('replace')):   0%|          | 0/1 [00:00<?, ?it/s]
                                                                          

Animation 2: FadeOut(Text('replace')):   0%|          | 0/1 [00:00<?, ?it/s]
                                                                            

Animation 3: FadeIn(Book of 3 submobjects):   0%|          | 0/1 [00:00<?, ?it/s]
                                                                                 

Animation 4: _MethodAnimation(Editor of 2 submobjects):   0%|          | 0/1 [00:00<?, ?it/s]
                                                                                             

Animation 5: _MethodAnimation(Editor of 2 submobjects):   0%|          | 0/1 [00:00<?, ?it/s]
                                                                                             

Animation 6: Write(Text("The editor's task is to replace 'John' with 'Jack'.")):   0%|          | 0/1 [00:00<?, ?it/s]
                               

In [13]:
error = manim_provider.fix_code(error)
print(error)

There was an error during execution.


Animation 0: Write(Text('append')):   0%|          | 0/1 [00:00<?, ?it/s]
                                                                         

Animation 2: FadeOut(Text('append')):   0%|          | 0/1 [00:00<?, ?it/s]
                                                                           

Animation 3: FadeIn(Scrapbook of 1 submobjects), etc.:   0%|          | 0/30 [00:00<?, ?it/s]
Animation 3: FadeIn(Scrapbook of 1 submobjects), etc.:  30%|███       | 9/30 [00:00<00:00, 86.55it/s]
Animation 3: FadeIn(Scrapbook of 1 submobjects), etc.:  60%|██████    | 18/30 [00:00<00:00, 86.74it/s]
Animation 3: FadeIn(Scrapbook of 1 submobjects), etc.:  93%|█████████▎| 28/30 [00:00<00:00, 92.31it/s]
                                                                                                      

Animation 4: FadeIn(GlueStick of 2 submobjects), etc.:   0%|          | 0/30 [00:00<?, ?it/s]
Animation 4: FadeIn(GlueStick of 2 submobjects), etc.:  60%